Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Load Data

In [ ]:
df = pd.read_csv("../data/ecommerce.csv")
df.head()

Basic Check

In [ ]:
df.shape
df.columns

define target and features

In [ ]:
y = df['cart_abandoned']

feature_cols = [
    'device_type',
    'user_type',
    'marketing_channel',
    'product_category',
    'unit_price',
    'quantity',
    'discount_percent',
    'discount_amount',
    'pages_viewed',
    'time_on_site_sec',
    'rating',
    'review_helpful_votes',
    'payment_method',
    'visit_day',
    'visit_month',
    'visit_weekday',
    'visit_season',
    'location',
    'session_duration_bucket'
]

X = df[feature_cols]

ENCODING

In [ ]:
X = pd.get_dummies(X, columns=['session_duration_bucket'], drop_first=True)

check class balance

In [ ]:
y.value_counts()

FEATURE ENGINEERING

In [ ]:
# ===== FEATURE ENGINEERING =====

# engagement level
df['engagement_score'] = df['pages_viewed'] * df['time_on_site_sec']

# discount impact
df['discount_ratio'] = df['discount_amount'] / (df['unit_price'] * df['quantity'] + 1)

# browsing efficiency
df['pages_per_second'] = df['pages_viewed'] / (df['time_on_site_sec'] + 1)

# session intensity
df['interaction_score'] = df['pages_viewed'] + df['added_to_cart'] * 2

# revisit behavior proxy
df['activity_intensity'] = df['time_on_site_sec'] / (df['visit_day'] + 1)

# update feature list
feature_cols += [
    'engagement_score',
    'discount_ratio',
    'pages_per_second',
    'interaction_score',
    'activity_intensity'
]

# rebuild X
X = df[feature_cols]

train-test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=5000, class_weight='balanced'),
    
    "Random Forest": RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42
),
    
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
}

TRAIN + EVALUATE

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    
    # 🔥 threshold tuning (important)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:,1]
        y_pred = model.predict(X_test)
    else:
        y_pred = model.predict(X_test)

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0)
    }

results_df = pd.DataFrame(results).T.sort_values("F1-score", ascending=False)
results_df

plot comparison

In [ ]:
results_df.plot(kind='bar', figsize=(10,6))
plt.title("Cart Abandonment Model Comparison (Improved)")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

BEST MODEL ANALYSIS

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

best_model_name = results_df.index[0]
best_model = models[best_model_name]

best_model.fit(X_train, y_train)

y_prob = best_model.predict_proba(X_test)[:,1]
y_pred = (y_prob > 0.4).astype(int)

print("Best Model:", best_model_name)

confusion matrix and ROC for best model

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

best_model = models["Random Forest"]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot()
plt.title("Confusion Matrix - Random Forest")
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Random Forest")
plt.legend()
plt.show()

In [ ]:
import joblib
import os

os.makedirs("../models", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

joblib.dump(best_model, "../models/cart_abandonment_best_model.pkl")
results_df.to_csv("../reports/cart_abandonment_metrics.csv", index=False)

## 📌 Observation

Despite applying multiple models and feature engineering techniques, cart abandonment prediction showed limited performance improvement.

## 📌 Reason

The dataset lacks sequential behavioral signals and user intent indicators, making the problem inherently difficult.

## 📌 Conclusion

Cart abandonment prediction requires richer behavioral tracking such as session sequences and historical patterns.